# Notebook 04: Feature Engineering & Data Preprocessing

## Project: Indian Air Quality Index (AQI) Comprehensive Analysis
## BTech Final Year Project - Data Science & Machine Learning (8th Semester)

### Objective:
Create advanced features, handle missing values, encode categorical variables, scale features, and prepare data for machine learning models.

### Prerequisites:
- Complete Notebook 01 (generates city_day_cleaned.csv)
- Libraries: pandas, numpy, scikit-learn

### Run Time: 10-15 minutes

## Step 1: Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported!')

Libraries imported!


### Explanation:

- **from sklearn.impute import KNNImputer**: Imputes missing values using K-Nearest Neighbors algorithm (finds similar rows and uses their values).

- **from sklearn.preprocessing import StandardScaler**: Scales features to have mean=0 and variance=1 (standard normal distribution).

- **from sklearn.preprocessing import MinMaxScaler**: Scales features to range [0, 1] (preserves original distribution shape).

- **from sklearn.preprocessing import LabelEncoder**: Converts categorical text labels to numeric codes (e.g., 'Good'=0, 'Poor'=1).

- **from sklearn.model_selection import train_test_split**: Splits data into training and testing sets for model evaluation.

In [2]:
data_path = os.path.join('..', 'datasets', 'city_day_cleaned.csv')
df = pd.read_csv(data_path, parse_dates=['Datetime'])
print(f'Loaded {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Date range: {df["Datetime"].min()} to {df["Datetime"].max()}')

Loaded 442985 rows, 18 columns
Date range: 2015-01-01 00:00:00 to 2025-07-31 00:00:00


## Step 2: Create Time-Based Features

In [3]:
df['Year'] = df['Datetime'].dt.year
df['Month'] = df['Datetime'].dt.month
df['Day'] = df['Datetime'].dt.day
df['DayOfWeek'] = df['Datetime'].dt.dayofweek
df['Quarter'] = df['Datetime'].dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['Month_Sin'] = np.sin(2 * np.pi * df['Month']/12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month']/12)
df['Day_Sin'] = np.sin(2 * np.pi * df['DayOfWeek']/7)
df['Day_Cos'] = np.cos(2 * np.pi * df['DayOfWeek']/7)
print('Time-based features created!')
print(df[['Datetime', 'Year', 'Month', 'DayOfWeek', 'IsWeekend', 'Month_Sin', 'Month_Cos']].head())

Time-based features created!
    Datetime  Year  Month  DayOfWeek  IsWeekend  Month_Sin  Month_Cos
0 2020-11-09  2020     11          0          0       -0.5   0.866025
1 2020-11-10  2020     11          1          0       -0.5   0.866025
2 2020-11-11  2020     11          2          0       -0.5   0.866025
3 2020-11-12  2020     11          3          0       -0.5   0.866025
4 2020-11-13  2020     11          4          0       -0.5   0.866025


### Explanation:

- **IsWeekend**: Binary flag (1=Saturday/Sunday, 0=weekday). Captures weekend pollution patterns.

- **Month_Sin/Cos**: Cyclical encoding of month using sine/cosine. Preserves circular nature (Dec close to Jan).

- **Day_Sin/Cos**: Cyclical encoding of day of week. Same circular preservation.

- **np.sin(2 * np.pi * x/period)**: Converts linear values to circular coordinates using trigonometric functions.

## Step 3: Create Lag Features (Previous Days' AQI)

In [4]:
df_sorted = df.sort_values(['City', 'Datetime']).reset_index(drop=True)
df_sorted['AQI_Lag1'] = df_sorted.groupby('City')['AQI'].shift(1)
df_sorted['AQI_Lag2'] = df_sorted.groupby('City')['AQI'].shift(2)
df_sorted['AQI_Lag3'] = df_sorted.groupby('City')['AQI'].shift(3)
df_sorted['AQI_Lag7'] = df_sorted.groupby('City')['AQI'].shift(7)
print('Lag features created!')
print(f'Rows with lag features: {df_sorted[["AQI_Lag1", "AQI_Lag2", "AQI_Lag3", "AQI_Lag7"]].isna().sum().to_dict()}')

Lag features created!
Rows with lag features: {'AQI_Lag1': 302, 'AQI_Lag2': 603, 'AQI_Lag3': 902, 'AQI_Lag7': 2098}


### Explanation:

- **groupby('City')['AQI'].shift(1)**: Creates previous day's AQI for each city separately.

- **shift(1)**: Shifts values down by 1 row (yesterday's value).

- **shift(7)**: Creates AQI from 7 days ago (same day last week).

- **Lag features**: Capture temporal dependencies (today's AQI depends on yesterday's).

## Step 4: Create Rolling Window Features

In [5]:
df_sorted['AQI_Rolling_Mean_3'] = df_sorted.groupby('City')['AQI'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_sorted['AQI_Rolling_Mean_7'] = df_sorted.groupby('City')['AQI'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_sorted['AQI_Rolling_Std_7'] = df_sorted.groupby('City')['AQI'].transform(lambda x: x.rolling(7, min_periods=1).std())
df_sorted['AQI_Rolling_Max_7'] = df_sorted.groupby('City')['AQI'].transform(lambda x: x.rolling(7, min_periods=1).max())
print('Rolling features created!')

Rolling features created!


### Explanation:

- **transform(lambda x: x.rolling(3).mean())**: Calculates 3-day rolling mean for each city.

- **min_periods=1**: Allows calculation even if fewer than 3 values available (uses what's available).

- **Rolling_Mean_3**: 3-day moving average (smooths short-term noise).

- **Rolling_Mean_7**: 7-day moving average (weekly trend).

- **Rolling_Std_7**: 7-day standard deviation (volatility measure).

- **Rolling_Max_7**: Maximum AQI in past 7 days (peak pollution).

## Step 5: Create Pollution Ratio Features

In [6]:
df_sorted['PM25_PM10_Ratio'] = df_sorted['PM2.5'] / (df_sorted['PM10'] + 1)
df_sorted['NO_NO2_Ratio'] = df_sorted['NO'] / (df_sorted['NO2'] + 1)
df_sorted['Pollution_Load'] = (df_sorted['PM2.5'] + df_sorted['PM10'] + df_sorted['NO2']) / 3
df_sorted['Oxidant_Level'] = (df_sorted['O3'] + df_sorted['NO2']) / 2
print('Pollution ratio features created!')

Pollution ratio features created!


### Explanation:

- **PM25_PM10_Ratio**: Indicates particle size distribution (high ratio = more fine particles).

- **+1 in denominator**: Prevents division by zero error.

- **NO_NO2_Ratio**: Indicates fresh vs aged emissions (high ratio = fresh vehicle emissions).

- **Pollution_Load**: Average of primary pollutants (overall pollution burden).

- **Oxidant_Level**: Average of oxidizing pollutants (chemical reactivity measure).

## Step 6: Encode Categorical Features

In [7]:
df_encoded = df_sorted.copy()
label_encoders = {}
for col in ['City', 'State', 'Region', 'AQI_Bucket']:
    le = LabelEncoder()
    df_encoded[f'{col}_Encoded'] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {len(le.classes_)} unique values encoded')
print('Categorical encoding completed!')

City: 302 unique values encoded

State: 35 unique values encoded
Region: 8 unique values encoded
AQI_Bucket: 7 unique values encoded
Categorical encoding completed!
State: 35 unique values encoded
Region: 8 unique values encoded
AQI_Bucket: 7 unique values encoded
Categorical encoding completed!


### Explanation:

- **LabelEncoder()**: Converts text categories to numbers (alphabetically: A=0, B=1, C=2).

- **fit_transform()**: Learns unique categories AND transforms them to numbers in one step.

- **astype(str)**: Ensures all values are strings before encoding (handles mixed types).

- **label_encoders dictionary**: Stores encoders for future use (converting back to text).

## Step 7: Handle Missing Values with KNN Imputer

In [ ]:
impute_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']

print('Creating 20% stratified sample for efficient KNN imputation...')
sample_size = int(len(df_encoded) * 0.2)
print(f'Sample size: {sample_size} rows (from {len(df_encoded)} total)')
print(f'This will take approximately 10-15 minutes...')

df_sample = df_encoded.sample(n=sample_size, random_state=42)

print('Fitting KNN Imputer on sample...')
imputer = KNNImputer(n_neighbors=5)
df_sample[impute_cols] = imputer.fit_transform(df_sample[impute_cols])

print('Applying imputation to full dataset...')
df_encoded.update(df_sample)

print(f'\nMissing values before: {df_sorted[impute_cols].isna().sum().sum()}')
print(f'Missing values after: {df_encoded[impute_cols].isna().sum().sum()}')
print('KNN imputation completed successfully!')

KeyboardInterrupt: 

### Explanation:

- **KNNImputer(n_neighbors=5)**: Finds 5 most similar rows and uses their average to fill missing values.

- **fit_transform()**: Learns patterns from data AND applies imputation.

- **Advantage over mean imputation**: Preserves relationships between features (smarter filling).

## Step 8: Remove Rows with Missing Target

In [ ]:
df_final = df_encoded.dropna(subset=['AQI', 'AQI_Bucket_Encoded', 'AQI_Lag1']).copy()
print(f'Rows after removing missing targets: {len(df_final)}')
print(f'Removed {len(df_encoded) - len(df_final)} rows with missing AQI or lag features')

### Explanation:

- **dropna(subset=['AQI', ...])**: Removes rows where AQI or specified columns are missing.

- **Cannot impute target**: We need actual AQI values for supervised learning (can't predict unknown).

- **AQI_Lag1 removal**: First row for each city has no previous day (lag is NaN).

## Step 9: Feature Scaling - StandardScaler

In [ ]:
feature_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3',
                'Month', 'DayOfWeek', 'IsWeekend', 'Month_Sin', 'Month_Cos', 'Day_Sin', 'Day_Cos',
                'AQI_Lag1', 'AQI_Lag2', 'AQI_Lag3', 'AQI_Lag7',
                'AQI_Rolling_Mean_3', 'AQI_Rolling_Mean_7', 'AQI_Rolling_Std_7', 'AQI_Rolling_Max_7',
                'PM25_PM10_Ratio', 'NO_NO2_Ratio', 'Pollution_Load', 'Oxidant_Level',
                'City_Encoded', 'State_Encoded', 'Region_Encoded']
scaler = StandardScaler()
df_final[feature_cols] = scaler.fit_transform(df_final[feature_cols])
print('Features scaled using StandardScaler!')
print(f'Feature mean after scaling: {df_final[feature_cols].mean().mean():.6f}')
print(f'Feature std after scaling: {df_final[feature_cols].std().mean():.6f}')

### Explanation:

- **StandardScaler()**: Transforms features to have mean=0, standard deviation=1.

- **Why scale?**: ML models perform better when all features have similar ranges.

- **fit_transform()**: Learns mean/std from data AND applies scaling transformation.

- **Mean ≈ 0, Std ≈ 1**: Confirms scaling worked correctly.

## Step 10: Prepare Train/Test Split

In [ ]:
X = df_final[feature_cols]
y_reg = df_final['AQI']
y_clf = df_final['AQI_Bucket_Encoded']
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Train/Test split: {X_train.shape[0]/(X_train.shape[0]+X_test.shape[0])*100:.0f}%/{X_test.shape[0]/(X_train.shape[0]+X_test.shape[0])*100:.0f}%')

### Explanation:

- **X**: Feature matrix (all input variables for ML models).

- **y_reg**: Target variable for regression (continuous AQI values).

- **y_clf**: Target variable for classification (AQI category encoded as numbers).

- **test_size=0.2**: Uses 20% of data for testing, 80% for training.

- **random_state=42**: Ensures reproducible split (same train/test sets every time).

- **stratify=y_clf**: Ensures train/test sets have same proportion of AQI categories.

## Step 11: Save Processed Data

In [ ]:
df_final.to_csv(os.path.join('..', 'datasets', 'city_day_ml_ready.csv'), index=False)
X_train.to_csv(os.path.join('..', 'datasets', 'X_train.csv'), index=False)
X_test.to_csv(os.path.join('..', 'datasets', 'X_test.csv'), index=False)
y_train_reg.to_csv(os.path.join('..', 'datasets', 'y_train_reg.csv'), index=False)
y_test_reg.to_csv(os.path.join('..', 'datasets', 'y_test_reg.csv'), index=False)
y_train_clf.to_csv(os.path.join('..', 'datasets', 'y_train_clf.csv'), index=False)
y_test_clf.to_csv(os.path.join('..', 'datasets', 'y_test_clf.csv'), index=False)
print('All processed data saved!')
print('READY FOR NOTEBOOK 05 (Regression Models)')

## Summary

Completed feature engineering:
1. Time-based features (year, month, day, cyclical encoding)
2. Lag features (1, 2, 3, 7 days)
3. Rolling window features (mean, std, max)
4. Pollution ratio features
5. Label encoding for categorical variables
6. KNN imputation for missing values
7. StandardScaler for feature normalization
8. Train/test split (80/20) with stratification

**Output Files:**
- city_day_ml_ready.csv: Complete ML-ready dataset
- X_train.csv, X_test.csv: Feature matrices
- y_train_reg.csv, y_test_reg.csv: Regression targets
- y_train_clf.csv, y_test_clf.csv: Classification targets

**Next**: Notebook 05 - ML Regression Models (AQI Prediction)